In [0]:
%run ../config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType


# Ensure Branches Silver Table Exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_catalog}.branches_clean (
    branch_id STRING NOT NULL,
    region STRING,
    city STRING,
    town STRING,
    branch_town STRING,
    latitude DOUBLE,
    longitude DOUBLE
)          
""")

# Read Bronze data
df = spark.read.table(f"{bronze_catalog}.branches").filter(F.col("branch_id").isNotNull())

# Silver Layer Transformation
df = (
    df
    # Clean strings
    .withColumn("town", F.initcap(F.trim(F.col("town"))))
    .withColumn("branch_town", F.initcap(F.trim(F.col("branch_town"))))

    # Change data type for lat and lon
    .withColumn("lat", F.col("lat").cast("double"))
    .withColumn("lon", F.col("lat").cast("double"))
)

# Rename columns
RENAME_MAP = {
    "lat": "latitude",
    "lon": "longitude"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Final schema
silver_df = (
    df.select(
        "branch_id",
        "region",
        "city",
        "town",
        "branch_town",
        "latitude",
        "longitude"
    )
)

# Write to silver table
(
silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{silver_catalog}.branches_clean")
)


# Validate that the Silver table was written successfully
count = spark.table(f"{silver_catalog}.branches_clean").count()

# Ensure table is not empty
assert count > 0, "branches_clean table is empty"

# Log result to job output
print(f"branches_clean validation passed: {count:,} records")